In [79]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.distributions.empirical_distribution import ECDF

# import shutil
# import re
# import os

# Ahora importa el módulo
from RRAM.Representate import config_ax, setup_paper_plt
    
setup_paper_plt(plt, latex=True, scaling=2)

In [80]:
ruta_datos = Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM"
ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Set"

In [81]:
results_path = ruta_figuras / "V_set_simulacion.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_creacion_1", "f8"),  # Float de 64 bits
        ("V_creacion_2", "f8"),  # Float de 64 bits
        ("V_creacion_3", "f8"),  # Float de 64 bits
        ("V_creacion_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_creacion_1",
        "V_creacion_2",
        "V_creacion_3",
        "V_creacion_4",
    ],  # NO se refiere a que se ha creado el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim = pd.DataFrame(resultados_txt)

df_resultados_sim["Numero"] = (
    df_resultados_sim["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

print(df_resultados_sim)

                    Archivo  V_creacion_1  V_creacion_2  V_creacion_3  \
0      log_simulacion_1.log        0.4879        0.9965        0.9967   
1     log_simulacion_10.log        0.5330        0.7627        0.7965   
2    log_simulacion_100.log        0.5188        0.8067        0.8330   
3    log_simulacion_101.log        0.4794        0.6525        0.7500   
4    log_simulacion_102.log        0.5032        0.7351        0.8421   
..                      ...           ...           ...           ...   
290   log_simulacion_95.log        0.5118        0.8992        0.9366   
291   log_simulacion_96.log        0.5540        0.6794        0.8497   
292   log_simulacion_97.log        0.5340        0.6850        0.8729   
293   log_simulacion_98.log        0.4917        0.8019        0.9125   
294   log_simulacion_99.log        0.5484        0.7666        0.8120   

     V_creacion_4  Numero  
0          1.0529       1  
1          0.8217      10  
2          0.9563     100  
3          

In [82]:
results_path = ruta_figuras / "V_set_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_set_derivada", "f8"),  # Float de 64 bits
        ("V_set_elbow", "f8"),
    ],
    encoding="utf-8",
    names=["Archivo", "V_set_derivada_V", "V_set_elbow_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados_exp = pd.DataFrame(resultados_txt)

df_resultados_exp["Numero"] = (
    df_resultados_exp["Archivo"].str.extract(r"Cycle_p_(\d+)", expand=False).astype(int)
)

print(df_resultados_exp)

          Archivo  V_set_derivada_V  V_set_elbow_V  Numero
0    Cycle_p_1000              0.53           0.52    1000
1    Cycle_p_1001              0.50           0.54    1001
2    Cycle_p_1002              0.49           0.54    1002
3    Cycle_p_1003              0.47           0.47    1003
4    Cycle_p_1004              0.51           0.50    1004
..            ...               ...            ...     ...
290   Cycle_p_995              0.52           0.51     995
291   Cycle_p_996              0.55           0.53     996
292   Cycle_p_997              0.51           0.50     997
293   Cycle_p_998              0.52           0.52     998
294   Cycle_p_999              0.54           0.54     999

[295 rows x 4 columns]


In [83]:
V_reset_sim_CF_1 = df_resultados_sim["V_creacion_1"]
V_reset_sim_CF_2 = df_resultados_sim["V_creacion_2"]
V_reset_sim_CF_all = df_resultados_sim["V_creacion_4"]

V_reset_exp_der = df_resultados_exp["V_set_derivada_V"]
V_reset_max_int = df_resultados_exp["V_set_elbow_V"]


# ECDF objects
ecdf_sim_CF_1 = ECDF(V_reset_sim_CF_1)
ecdf_sim_CF_2 = ECDF(V_reset_sim_CF_2)
ecdf_sim_CF_all = ECDF(V_reset_sim_CF_all)

ecdf_exp_der = ECDF(V_reset_exp_der)
ecdf_exp_max_int = ECDF(V_reset_max_int)

fig, ax = plt.subplots(figsize=(12, 9))
config_ax(ax)

# Common range
min_val = min(
    min(V_reset_sim_CF_1),
    min(V_reset_sim_CF_2),
    min(V_reset_sim_CF_all),
    min(V_reset_exp_der),
    min(V_reset_max_int),
)
max_val = max(
    max(V_reset_sim_CF_1),
    max(V_reset_sim_CF_2),
    max(V_reset_sim_CF_all),
    max(V_reset_exp_der),
    max(V_reset_max_int),
)
x = np.linspace(min_val, max_val, 1000)

# Evaluate ECDFs
y_sim_1 = ecdf_sim_CF_1(x)
y_sim_2 = ecdf_sim_CF_2(x)
y_sim_all = ecdf_sim_CF_all(x)
y_der = ecdf_exp_der(x)
y_max_int = ecdf_exp_max_int(x)

# Plot ECDFs
ax.step(x, y_sim_1, where="post", label="Simulation - 1 CF")
# ax.step(x, y_sim_2, where="post", label="Simulation - 2 CF")
# ax.step(x, y_sim_all, where="post", label="Simulation - All CF")
ax.step(x, y_der, where="post", label="Experiment - Derivative")
ax.step(x, y_max_int, where="post", label="Experiment - Elbow")

# Labels in English
ax.set_xlabel(r"Set voltage (\si{\volt})")
ax.set_ylabel("CDF")
ax.grid(True)
ax.legend()

fig.savefig(str(Path.cwd()) + "/CDF_set_combinado.pdf", dpi=300, bbox_inches="tight")
plt.close(fig)

In [84]:
ruta_datos = Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM"
ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Reset"


In [85]:
# Obtener V_set de los archivos de simulación
results_path = ruta_figuras / "V_reset_simulacion.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_rotura_1", "f8"),  # Float de 64 bits
        ("V_rotura_2", "f8"),  # Float de 64 bits
        ("V_rotura_3", "f8"),  # Float de 64 bits
        ("V_rotura_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_rotura_1",
        "V_rotura_2",
        "V_rotura_3",
        "V_rotura_4",
    ],  # NO se refiere a que se ha roto el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim = pd.DataFrame(resultados_txt)
df_resultados_sim["Numero"] = (
    df_resultados_sim["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

# Elimino la primera columna del dataframe
df_resultados_sim = df_resultados_sim.drop(columns=["Archivo"])

# Muevo la primera columna al de numero al inicio
cols = df_resultados_sim.columns.tolist()
cols = [cols[-1]] + cols[:-1]
df_resultados_sim = df_resultados_sim[cols]
print(df_resultados_sim)


     Numero  V_rotura_1  V_rotura_2  V_rotura_3  V_rotura_4
0         1     -0.7623     -1.1827     -1.2775     -1.3644
1        10     -1.1369     -1.1418     -1.2611     -1.3572
2       100     -0.7384     -1.1190     -1.2663     -1.3271
3       101     -0.8742     -1.1591     -1.2600     -1.2785
4       102     -1.0675     -1.1130     -1.3188     -1.3703
..      ...         ...         ...         ...         ...
290      95     -0.9008     -1.1673     -1.1799     -1.2851
291      96     -1.1402     -1.1697     -1.2137     -1.3495
292      97     -1.1168     -1.1649     -1.1803     -1.3332
293      98     -0.9334     -1.1166     -1.1266     -1.3076
294      99     -1.0779     -1.1508     -1.2102     -1.2400

[295 rows x 5 columns]


In [86]:
results_path = ruta_figuras / "V_Reset_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_reset_derivada", "f8"),  # Float de 64 bits
        ("V_reset_max_intensidad", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=["Archivo", "V_reset_derivada_V", "V_reset_max_intensidad_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados = pd.DataFrame(resultados_txt)

df_resultados["Numero"] = (
    df_resultados["Archivo"].str.extract(r"Cycle_n_(\d+)", expand=False).astype(int)
)
print(df_resultados)


          Archivo  V_reset_derivada_V  V_reset_max_intensidad_V  Numero
0    Cycle_n_1000               -1.14                     -1.07    1000
1    Cycle_n_1001               -1.13                     -1.07    1001
2    Cycle_n_1002               -1.12                     -1.04    1002
3    Cycle_n_1003               -1.14                     -1.07    1003
4    Cycle_n_1004               -1.12                     -1.05    1004
..            ...                 ...                       ...     ...
290   Cycle_n_995               -1.13                     -1.06     995
291   Cycle_n_996               -1.15                     -1.07     996
292   Cycle_n_997               -1.15                     -1.09     997
293   Cycle_n_998               -1.14                     -1.08     998
294   Cycle_n_999               -1.14                     -1.07     999

[295 rows x 4 columns]


In [87]:
V_reset_sim_CF_1 = df_resultados_sim["V_rotura_1"]
V_reset_sim_CF_2 = df_resultados_sim["V_rotura_2"]
V_reset_sim_CF_all = df_resultados_sim["V_rotura_4"]

V_reset_exp_der = df_resultados["V_reset_derivada_V"]
V_reset_max_int = df_resultados["V_reset_max_intensidad_V"]

# ECDF objects
ecdf_sim_CF_1 = ECDF(V_reset_sim_CF_1)
ecdf_sim_CF_2 = ECDF(V_reset_sim_CF_2)
ecdf_sim_CF_all = ECDF(V_reset_sim_CF_all)

ecdf_exp_der = ECDF(V_reset_exp_der)
ecdf_exp_max_int = ECDF(V_reset_max_int)

fig, ax = plt.subplots(figsize=(12, 9))
config_ax(ax)

# Common range
min_val = min(
    min(V_reset_sim_CF_1),
    min(V_reset_sim_CF_2),
    min(V_reset_sim_CF_all),
    min(V_reset_exp_der),
    min(V_reset_max_int),
)
max_val = max(
    max(V_reset_sim_CF_1),
    max(V_reset_sim_CF_2),
    max(V_reset_sim_CF_all),
    max(V_reset_exp_der),
    max(V_reset_max_int),
)
x = np.linspace(min_val, max_val, 1000)

# Evaluate ECDFs
y_sim_1 = ecdf_sim_CF_1(x)
y_sim_2 = ecdf_sim_CF_2(x)
y_sim_all = ecdf_sim_CF_all(x)
y_der = ecdf_exp_der(x)
y_max_int = ecdf_exp_max_int(x)

# Plot ECDFs
ax.step(x, y_sim_1, where="post", label="Simulation - 1 CF")
ax.step(x, y_sim_2, where="post", label="Simulation - 2 CF")
ax.step(x, y_sim_all, where="post", label="Simulation - All CF")
ax.step(x, y_der, where="post", label="Experiment - Derivative")
ax.step(x, y_max_int, where="post", label="Experiment - Max Current")

# Labels in English
ax.set_xlabel(r"Reset voltage (\si{\volt})")
ax.set_ylabel("CDF")
ax.grid(True)
ax.legend()

fig.savefig(str(Path.cwd()) + "/CDF_reset_combinado.pdf", dpi=300, bbox_inches="tight")
plt.close(fig)
